In [ ]:
import math
import pandas as pd
import numpy as np
import wandb

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.context_scaling.eval_index import load_eval_index

### functions

In [ ]:
def joint_tasks_plot_fetch(
    idx: pd.DataFrame,
    task_keys: list,
    group_col: str,
    project: str = "ideas_cv/llm-random-test",
):
    records = []
    api = wandb.Api()

    for _, row in idx.iterrows():
        run_id = row["run_id"]
        run = api.run(f"{project}/{run_id}")

        hist = run.history(
            keys=["steps/eval/loss"] + task_keys,
            pandas=True,
        )

        hist = hist.dropna(subset=["steps/eval/loss"]).copy()
        hist["_step"] = hist["_step"] - 1

        group_value = row[group_col]
        dmodel = row.get("common.dmodel")
        dff = row.get("common.dff")
        sequence_length = row.get("common.sequence_length")
        attn_type = "MQA" if row.get("common.kv_heads") == 1 else "MHA"

        for _, hrow in hist.iterrows():
            loss = hrow["steps/eval/loss"]
            step = hrow["_step"]

            for task_key in task_keys:
                if task_key in hrow and pd.notna(hrow[task_key]):
                    records.append(
                        {
                            "run_id": run_id,
                            "group": group_value,
                            "step": step,
                            "loss": loss,
                            "task": task_key,
                            "task_value": hrow[task_key],
                            "dmodel": dmodel,
                            "dff": dff,
                            "attn_type": attn_type,
                            "sequence_length": sequence_length,
                        }
                    )

    return pd.DataFrame(records)


def plot_joint_tasks(
    plot_df: pd.DataFrame,
    task_keys: list,
    plot_title: str,
    symbol_col: str | None = "attn_type",
    symbol_map: dict | None = None,
    fit_lines=False,
    height=None,
    width=1400,
    marker_size=6,
):
    plot_df = plot_df.copy()
    plot_df["group"] = plot_df["group"].astype(str)
    plot_df["loss"] = plot_df["loss"].astype(float)
    plot_df["task_value"] = plot_df["task_value"].astype(float)

    if symbol_map is None:
        symbol_map = {
            "MHA": "circle",
            "MQA": "x",
        }

    perp_mask = plot_df["task"] == "steps/eval/lambada_standard/perplexity"
    plot_df.loc[perp_mask, "task_value"] = np.log(plot_df.loc[perp_mask, "task_value"])

    plot_df["task_label"] = (
        plot_df["task"]
        .str.replace("steps/eval/", "", regex=False)
        .str.replace("lambada_standard/perplexity", "lambada (log perplexity)", regex=False)
        .str.replace("lambada_standard/acc", "lambada (acc)", regex=False)
        .str.replace("/acc_norm", "", regex=False)
        .str.replace("/acc", "", regex=False)
    )

    task_order = (
        plot_df[["task", "task_label"]]
        .drop_duplicates()
        .set_index("task")["task_label"]
        .reindex(task_keys)
        .tolist()
    )

    group_order = sorted(plot_df["group"].dropna().unique(), key=lambda x: (len(x), x))
    colors = px.colors.qualitative.Plotly
    color_map = {g: colors[i % len(colors)] for i, g in enumerate(group_order)}

    n_tasks = len(task_order)
    ncols = 2
    nrows = math.ceil(n_tasks / ncols)

    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=task_order,
        horizontal_spacing=0.06,
        vertical_spacing=0.03,
    )

    shown_legend = set()

    for i, task_label in enumerate(task_order):
        row = i // ncols + 1
        col = i % ncols + 1

        task_df = plot_df[plot_df["task_label"] == task_label]

        for group in group_order:
            group_df = task_df[task_df["group"] == group].copy()
            if group_df.empty:
                continue

            color = color_map[group]

            if symbol_col and symbol_col in group_df.columns:
                symbol_values = list(group_df[symbol_col].dropna().unique())
                if not symbol_values:
                    symbol_values = [None]
            else:
                symbol_values = [None]

            for symbol_value in symbol_values:
                if symbol_value is None:
                    sub = group_df.copy()
                else:
                    sub = group_df[group_df[symbol_col] == symbol_value].copy()

                if sub.empty:
                    continue

                marker_symbol = symbol_map.get(symbol_value, "circle") if symbol_value is not None else "circle"
                legend_key = (group, symbol_value)

                customdata_cols = ["dmodel", "step"]
                hover_lines = [
                    "loss=%{x}",
                    "value=%{y}",
                    "dmodel=%{customdata[0]}",
                    "step=%{customdata[1]}",
                ]

                if symbol_col and symbol_col in sub.columns:
                    customdata_cols.append(symbol_col)
                    hover_lines.insert(3, f"{symbol_col}=%{{customdata[2]}}")

                customdata = np.stack(
                    [sub[c].astype(object).to_numpy() for c in customdata_cols],
                    axis=-1,
                )

                trace_name = group if symbol_value is None else f"{group} / {symbol_value}"

                fig.add_trace(
                    go.Scatter(
                        x=sub["loss"],
                        y=sub["task_value"],
                        mode="markers",
                        name=trace_name,
                        legendgroup=trace_name,
                        showlegend=legend_key not in shown_legend,
                        marker=dict(
                            size=marker_size,
                            color=color,
                            symbol=marker_symbol,
                            opacity=0.8,
                        ),
                        customdata=customdata,
                        hovertemplate="<br>".join(hover_lines) + "<extra>%{fullData.name}</extra>",
                    ),
                    row=row,
                    col=col,
                )
                shown_legend.add(legend_key)

                if fit_lines and len(sub) >= 2:
                    x = sub["loss"].to_numpy()
                    y = sub["task_value"].to_numpy()
                    mask = np.isfinite(x) & np.isfinite(y)
                    x = x[mask]
                    y = y[mask]

                    if len(x) >= 2 and np.unique(x).size >= 2:
                        m, b = np.polyfit(x, y, 1)
                        xs = np.linspace(x.min(), x.max(), 100)
                        ys = m * xs + b

                        fig.add_trace(
                            go.Scatter(
                                x=xs,
                                y=ys,
                                mode="lines",
                                name=trace_name,
                                legendgroup=trace_name,
                                showlegend=False,
                                line=dict(color=color, width=2),
                                hoverinfo="skip",
                            ),
                            row=row,
                            col=col,
                        )

        fig.update_xaxes(title_text="Final Loss", row=row, col=col)
        fig.update_yaxes(title_text="", row=row, col=col)

    fig.update_layout(
        title=plot_title,
        width=width,
        height=height or 420 * nrows,
        margin=dict(l=10, r=10, t=10, b=10),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1.0,
        ),
    )

    fig.show()

### setup

In [ ]:
eval_dir = "/home/janek/Documents/IDEAS/2503_grid_namesfix"
idx = load_eval_index(eval_dir)
print(f"index shape: {idx.shape}")

task_keys = [
    "steps/eval/lambada_standard/perplexity",
    "steps/eval/lambada_standard/acc",
    "steps/eval/winogrande/acc",
    "steps/eval/social_iqa/acc",
    "steps/eval/sciq/acc_norm",
    "steps/eval/openbookqa/acc_norm",
    "steps/eval/hellaswag/acc_norm",
    "steps/eval/arc_easy/acc_norm",
    "steps/eval/arc_challenge/acc_norm",
    "steps/eval/piqa/acc_norm",
]

### MHA vs MQA

In [ ]:
plot_df = joint_tasks_plot_fetch(idx, task_keys, group_col="common.kv_heads")
plot_df["group"] = plot_df["group"].map(lambda x: "mqa" if x == 1 else "mha")

plot_joint_tasks(plot_df, task_keys, plot_title="MHA vs MQA")

### Sequence Length

In [ ]:
plot_df = joint_tasks_plot_fetch(idx, task_keys, group_col="common.sequence_length")

plot_joint_tasks(plot_df, task_keys, plot_title="Sequence length", fit_lines=False)

### Sequence Length = 1024 (decayed vs high LR)

In [ ]:
subset = idx[idx["common.sequence_length"] == 1024]
plot_df = joint_tasks_plot_fetch(subset, task_keys, group_col="common.sequence_length")
plot_df["group"] = np.where(plot_df["step"] == 320000, "decayed", "high LR")

plot_joint_tasks(
    plot_df, task_keys,
    plot_title="Sequence length: 1024",
    fit_lines=False,
    symbol_col="attn_type",
    symbol_map={"MHA": "circle", "MQA": "x"},
    marker_size=10,
)

### MHA vs MQA (seq_len=2048, matched dff)

In [ ]:
subset = idx[
    (idx["common.sequence_length"] == 2048) &
    ((idx["common.kv_heads"] != 1) | (idx["common.dmodel"] * 4 == idx["common.dff"]))
]

plot_df = joint_tasks_plot_fetch(subset, task_keys, group_col="common.kv_heads")
plot_df["group"] = plot_df["group"].map(lambda x: "mqa" if x == 1 else "mha")

allowed_steps = [32000 * (i + 1) for i in range(8)]
plot_df = plot_df[plot_df["step"].isin(allowed_steps)]

plot_joint_tasks(plot_df, task_keys, plot_title="MHA vs MQA, seq len = 2048", fit_lines=True)